# Olist Lakehouse — Exploratory Data Analysis (EDA) Profiling\n\n> **Tác giả:** Anh Tú (IT Lecturer & Technical Writer)  \n> **Phương pháp:** Pedagogical Approach — Chuyển hóa kiến thức phức tạp thành trực quan, dễ hiểu.  \n> **Môi trường:** Apache Spark 3.5.0 + PostgreSQL 16 + MinIO S3 (chạy trên container `olist_spark_notebook:8888`)  \n> **Bản quyền:** *Made by Anh Tu - Share to be share*  \n\n---\n\n## 💡 TẠI SAO DATA ENGINEER CẦN EDA CHUYÊN BIỆT?\nKhác với **Data Science EDA** (tìm kiếm tương quan `correlation`, phân phối chuẩn để huấn luyện mô hình ML), **Data Engineering EDA** là quá trình khám nghiệm cấu trúc và ranh giới dữ liệu (**Structural & Boundary Inspection**) nhằm:\n1. **Thiết lập Data Contract:** Định hình kích thước kiểu dữ liệu chính xác (VARCHAR sizing, tránh tràn bộ nhớ).\n2. **Xây dựng Data Quality Rule Catalog:** Không đặt bừa rule `NOT NULL` hay `> 0` làm sập pipeline (Pipeline Crash), mà xác lập đúng chính sách (*Quarantine, Log Warning, Unknown Value*).\n3. **Phát hiện dị thường xuyên nguồn (Cross-Source Isolation):** Đo lường tính toàn vẹn tham chiếu giữa RDBMS OLTP và Object Storage Landing.

## 1. Khởi Tạo SparkSession Phân Tán Kết Nối PostgreSQL JDBC & MinIO S3A\nSpark hoạt động như một Distributed Compute Engine đọc trực tiếp từ `s3a://landing/` và PostgreSQL pooler qua JDBC.

In [ ]:
import os\nimport json\nimport pandas as pd\nfrom pyspark.sql import SparkSession\n\n# Khởi tạo Spark Session với cấu hình S3A MinIO\nspark = SparkSession.builder \\\n    .appName("Olist-EDA-Profiling-5Pillars") \\\n    .config("spark.hadoop.fs.s3a.endpoint", os.getenv("MINIO_ENDPOINT", "http://minio:9000")) \\\n    .config("spark.hadoop.fs.s3a.access.key", os.getenv("AWS_ACCESS_KEY_ID", "minioadmin")) \\\n    .config("spark.hadoop.fs.s3a.secret.key", os.getenv("AWS_SECRET_ACCESS_KEY", "minioadmin")) \\\n    .config("spark.hadoop.fs.s3a.path.style.access", "true") \\\n    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \\\n    .getOrCreate()\n\nprint("✔ Spark Session đã khởi tạo thành công với S3A Filesystem Connector!")\n\n# Đọc file siêu dữ liệu tổng hợp\nwith open('../docs/week02/eda_summary.json', 'r') as f:\n    eda_summary = json.load(f)\n\n# Bảng tổng quan 10 thực thể từ 2 nguồn\noverview = [\n    {\n        'Thực Thể': name,\n        'Nguồn Lưu Trữ': info['source_type'],\n        'Vị Trí Gốc': info.get('source_origin', info.get('s3_uri')),\n        'Số Dòng': f"{info['row_count']:,}",\n        'Số Cột': info['column_count']\n    }\n    for name, info in eda_summary.items()\n]\npd.DataFrame(overview)

## 2. Trụ Cột 1: Schema Sizing & Storage Allocation (Tối Ưu Chuỗi)\nTrong cơ sở dữ liệu lớn, việc để `TEXT` hoặc `VARCHAR(255)` vô tội vạ làm lãng phí RAM của Spark khi shuffle dữ liệu. Thống kê `min_len` và `max_len` giúp định kích thước cột chính xác.

In [ ]:
sizing_rows = []\nfor entity, info in eda_summary.items():\n    for col, stat in info['columns'].items():\n        if stat.get('data_category') == 'string':\n            sizing_rows.append({\n                'Thực Thể': entity,\n                'Cột': col,\n                'Min Len': stat.get('min_len', 0),\n                'Max Len': stat.get('max_len', 0),\n                'Avg Len': stat.get('avg_len', 0.0),\n                'Khuyến Nghị Kiểu Dữ Liệu': f"VARCHAR({stat.get('max_len', 32)})" if stat.get('max_len', 0) > 0 else "VARCHAR(32)"\n            })\npd.DataFrame(sizing_rows).head(15)

## 3. Trụ Cột 2: Domain Boundaries & Dị Thường Giá Trị Số\nPhát hiện các trường số có giá trị âm hoặc bằng 0 (ví dụ: `payment_value == 0` do sử dụng voucher giảm 100%).

In [ ]:
numeric_rows = []\nfor entity, info in eda_summary.items():\n    for col, stat in info['columns'].items():\n        if stat.get('data_category') == 'numeric':\n            numeric_rows.append({\n                'Thực Thể': entity,\n                'Cột': col,\n                'Min Value': stat.get('min_value'),\n                'Max Value': stat.get('max_value'),\n                'Số Dòng = 0': stat.get('zero_count', 0),\n                'Số Dòng Âm (<0)': stat.get('negative_count', 0)\n            })\npd.DataFrame(numeric_rows)

## 4. Trụ Cột 3 & 4: Tính Nhất Quán Thời Gian & Phân Bố Khóa\nĐo lường độ bao phủ thời gian và phát hiện hiện tượng thổi phồng khóa (`customer_id` vs `customer_unique_id`).

In [ ]:
# Đọc trực tiếp từ Spark DataFrame để kiểm tra tỷ lệ trùng lặp khách hàng\ncust_info = eda_summary.get('customers', {})\ntotal_cust_ids = cust_info['columns']['customer_id']['distinct_count']\nunique_persons = cust_info['columns']['customer_unique_id']['distinct_count']\ndiff = total_cust_ids - unique_persons\npct_inflation = round((diff / total_cust_ids) * 100, 2)\nprint(f"Tổng số transaction customer_id : {total_cust_ids:,}")\nprint(f"Tổng số khách hàng thực tế     : {unique_persons:,}")\nprint(f"Số mã bị thổi phồng (Lặp lại) : {diff:,} ({pct_inflation}%)")

## 5. Trụ Cột 5: Cross-Source Relational Integrity Check\nKiểm tra xem `order_items` trong PostgreSQL có chứa `product_id` nào không tồn tại trong `s3a://landing/ref/olist_products_dataset.csv` hay không.

In [ ]:
# Minh họa đọc S3A qua PySpark API\ntry:\n    df_products = spark.read.option("header", "true").csv("s3a://landing/ref/olist_products_dataset.csv")\n    print(f"✔ Đọc thành công từ S3A MinIO! Số lượng sản phẩm: {df_products.count():,}")\nexcept Exception as e:\n    print("Chưa bật kết nối S3A trực tiếp trong container, sử dụng Boto3 metadata:")\n    prod_info = eda_summary.get('products', {})\n    print(f"Số lượng sản phẩm trong MinIO Landing S3: {prod_info.get('row_count'):,}")

\n---\n*Made by Anh Tu - Share to be share*